In [ ]:
%pip install dlt[rest_api] requests

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.stage")

In [ ]:
import dlt
import requests
import pandas as pd
from delta.tables import DeltaTable

In [ ]:
# ECB Statistical Data Warehouse REST API — free, no API key
# Full series key format on ECB portal: {DATASET}.{KEY}
# URL format: /data/{DATASET}/{KEY}
SERIES = {
    "ecb_deposit_rate": ("FM", "B.U2.EUR.4F.KR.DFR.LEV"),       # Deposit Facility Rate level
    "euribor_3m":       ("FM", "B.U2.EUR.RT.MM.EURIBOR3MD_.HSTA"),  # EURIBOR 3-month historical
    "euribor_6m":       ("FM", "B.U2.EUR.RT.MM.EURIBOR6MD_.HSTA"),  # EURIBOR 6-month historical
    "bond_10y":         ("YC", "B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y"),  # 10Y Euro area yield
}

def validate_series():
    """Quick probe of each series key — prints OK/FAIL before the full fetch."""
    base = "https://data-api.ecb.europa.eu/service/data"
    for name, (dataset, key) in SERIES.items():
        url = f"{base}/{dataset}/{key}"
        r = requests.get(url, params={"format": "jsondata", "startPeriod": "2024-01-01",
                                       "endPeriod": "2024-01-31", "detail": "dataonly"}, timeout=30)
        status = "OK  " if r.status_code == 200 else f"FAIL ({r.status_code})"
        print(f"  {status}  {dataset}/{key}  [{name}]")

print("Validating ECB series keys:")
validate_series()

In [ ]:
@dlt.resource(name="macro_rates", write_disposition="merge", primary_key=["date", "indicator"])
def fetch_ecb_rates():
    base = "https://data-api.ecb.europa.eu/service/data"
    for indicator_name, (dataset, series_key) in SERIES.items():
        url = f"{base}/{dataset}/{series_key}"
        r = requests.get(url, params={"format": "jsondata", "startPeriod": "2020-01-01"}, timeout=30)
        if r.status_code != 200:
            print(f"  SKIP {indicator_name}: {r.status_code} — {url}")
            continue
        payload = r.json()
        series_data = payload["dataSets"][0]["series"]
        series_idx = list(series_data.keys())[0]
        obs = series_data[series_idx]["observations"]
        periods = payload["structure"]["dimensions"]["observation"][0]["values"]
        for i, period in enumerate(periods):
            obs_val = obs.get(str(i))
            if obs_val and obs_val[0] is not None:
                yield {"date": period["id"], "indicator": indicator_name, "value": float(obs_val[0])}

records = list(fetch_ecb_rates())
print(f"Fetched {len(records)} ECB macro records")
if records:
    pd.DataFrame(records).groupby("indicator").size().reset_index(name="rows")